# Add simulation data to CAMELS-DE-1h

Build a file `CAMELS_DE_1h_simulations_benchmark.csv` with metadata about model runs for each catchment

In [1]:
from pathlib import Path
from glob import glob

import pandas as pd
from tqdm import tqdm

In [2]:
path_lstm = Path("../models/LSTM_simulation/")
path_hbv = Path("../models/HBV_simulation/")

# simulation periods
training_period = ["2004-01-01 01:00:00", "2019-12-31 23:00:00"]
validation_period = ["2001-01-01 01:00:00", "2003-12-31 23:00:00"]
testing_period = ["2020-01-01 01:00:00", "2024-12-31 23:00:00"]

In [3]:
# get camels_ids from hydromet timeseries
ids = [f.split("_")[-1].split(".")[0] for f in glob("../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_*.csv")]

# sort camels_ids
ids = sorted(ids)

print(f"Total number of stations in CAMELS-DE-1h: {len(ids)}")

Total number of stations in CAMELS-DE-1h: 1611


## Create CAMELS_DE_simulation_benchmark.csv

In [4]:
# dataframe to store results
df_results = pd.DataFrame(index=ids)

Calculate percentage of data available in training, validation, testing period for each catchment.

In [5]:
# read the timeseries
for id in tqdm(ids):
    df = pd.read_csv(f"../CAMELS-DE-1h/timeseries/CAMELS_DE_1h_hydromet_timeseries_{id}.csv", index_col=0, parse_dates=True)

    # count number with discharge_spec that are not NaN for each period
    training_count = df.loc[training_period[0]:training_period[1], "discharge_spec_obs"].count()
    validation_count = df.loc[validation_period[0]:validation_period[1], "discharge_spec_obs"].count()
    testing_count = df.loc[testing_period[0]:testing_period[1], "discharge_spec_obs"].count()

    # get ratio of stations with discharge_spec data for each period
    training_perc_complete = training_count / len(df.loc[training_period[0]:training_period[1], "discharge_spec_obs"]) * 100 # in percent
    validation_perc_complete = validation_count / len(df.loc[validation_period[0]:validation_period[1], "discharge_spec_obs"]) * 100 # in percent
    testing_perc_complete = testing_count / len(df.loc[testing_period[0]:testing_period[1], "discharge_spec_obs"]) * 100 # in percent

    # store results
    df_results.loc[id, "training_perc_complete"] = round(training_perc_complete, 2)
    df_results.loc[id, "validation_perc_complete"] = round(validation_perc_complete, 2)
    df_results.loc[id, "testing_perc_complete"] = round(testing_perc_complete, 2)

df_results

100%|██████████| 1611/1611 [08:47<00:00,  3.05it/s]


,training_perc_complete,validation_perc_complete,testing_perc_complete
DE110000,100.00,100.00,100.00
DE110010,96.52,95.14,97.33
DE110020,100.00,100.00,100.00
DE110030,100.00,100.00,100.00
DE110040,100.00,100.00,100.00
...,...,...,...
DEG11340,44.78,0.00,96.66
DEG11350,44.27,0.00,96.66
DEG11360,25.00,0.00,96.66
DEG11380,44.78,0.00,96.66


Add NSE LSTM & HBV

In [6]:
df_nse_lstm = pd.read_csv(path_lstm / "NSE_ensemble_results.csv")
df_nse_hbv = pd.read_csv(path_hbv / "NSE_ensemble_results.csv")

# set index to gauge_id
df_nse_lstm = df_nse_lstm.set_index("gauge_id")
df_nse_hbv = df_nse_hbv.set_index("gauge_id")

# rename columns
df_nse_lstm = df_nse_lstm.rename(columns={"NSE_ensemble": "NSE_lstm"})
df_nse_hbv = df_nse_hbv.rename(columns={"NSE_ensemble": "NSE_hbv"})

# join the two dataframes
df_results = df_results.merge(df_nse_lstm["NSE_lstm"], left_index=True, right_index=True, how="left")
df_results = df_results.merge(df_nse_hbv["NSE_hbv"], left_index=True, right_index=True, how="left")

# round NSE values to 3 decimal places
df_results[["NSE_lstm", "NSE_hbv"]] = df_results[["NSE_lstm", "NSE_hbv"]].round(3)

df_results

,training_perc_complete,validation_perc_complete,testing_perc_complete,NSE_lstm,NSE_hbv
DE110000,100.00,100.00,100.00,0.943,0.810
DE110010,96.52,95.14,97.33,0.900,0.315
DE110020,100.00,100.00,100.00,0.931,0.625
DE110030,100.00,100.00,100.00,0.883,0.539
DE110040,100.00,100.00,100.00,0.889,0.674
...,...,...,...,...,...
DEG11340,44.78,0.00,96.66,0.312,0.350
DEG11350,44.27,0.00,96.66,0.722,0.599
DEG11360,25.00,0.00,96.66,0.358,-0.004
DEG11380,44.78,0.00,96.66,0.788,0.655


In [7]:
# save
df_results.to_csv("../CAMELS-DE-1h/CAMELS_DE_1h_simulation_benchmark.csv", index_label="gauge_id")